In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
import matplotlib.pyplot as plt


def forward_selection(indep_X, dep_Y, n):
    """
    Greedy forward selection: starts with no features and adds one at a time,
    choosing the feature that improves accuracy the most, until n features are selected.
    Uses LogisticRegression as the evaluation estimator.
    """
    X = indep_X.values if hasattr(indep_X, 'values') else indep_X
    cols = list(range(X.shape[1]))
    selected = []
    remaining = cols.copy()

    X_train, X_test, y_train, y_test = train_test_split(
        X, dep_Y, test_size=0.25, random_state=0
    )

    for _ in range(n):
        best_acc = -np.inf
        best_col = None
        for col in remaining:
            candidate = selected + [col]
            clf = LogisticRegression(max_iter=1000, random_state=0)
            clf.fit(X_train[:, candidate], y_train)
            acc = accuracy_score(y_test, clf.predict(X_test[:, candidate]))
            if acc > best_acc:
                best_acc = acc
                best_col = col
        selected.append(best_col)
        remaining.remove(best_col)

    return X[:, selected], selected


def split_scalar(indep_X, dep_Y):
    X_train, X_test, y_train, y_test = train_test_split(
        indep_X, dep_Y, test_size=0.25, random_state=0
    )
    sc = StandardScaler()
    X_train = sc.fit_transform(X_train)
    X_test = sc.transform(X_test)
    return X_train, X_test, y_train, y_test


def cm_prediction(classifier, X_test):
    y_pred = classifier.predict(X_test)
    from sklearn.metrics import confusion_matrix, accuracy_score, classification_report
    cm = confusion_matrix(y_test, y_pred)
    Accuracy = accuracy_score(y_test, y_pred)
    report = classification_report(y_test, y_pred)
    return classifier, Accuracy, report, X_test, y_test, cm


def logistic(X_train, y_train, X_test):
    from sklearn.linear_model import LogisticRegression
    classifier = LogisticRegression(random_state=0, max_iter=1000)
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test)
    return classifier, Accuracy, report, X_test, y_test, cm


def svm_linear(X_train, y_train, X_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='linear', random_state=0)
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test)
    return classifier, Accuracy, report, X_test, y_test, cm


def svm_NL(X_train, y_train, X_test):
    from sklearn.svm import SVC
    classifier = SVC(kernel='rbf', random_state=0)
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test)
    return classifier, Accuracy, report, X_test, y_test, cm


def Navie(X_train, y_train, X_test):
    from sklearn.naive_bayes import GaussianNB
    classifier = GaussianNB()
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test)
    return classifier, Accuracy, report, X_test, y_test, cm


def knn(X_train, y_train, X_test):
    from sklearn.neighbors import KNeighborsClassifier
    classifier = KNeighborsClassifier(n_neighbors=5, metric='minkowski', p=2)
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test)
    return classifier, Accuracy, report, X_test, y_test, cm


def Decision(X_train, y_train, X_test):
    from sklearn.tree import DecisionTreeClassifier
    classifier = DecisionTreeClassifier(criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test)
    return classifier, Accuracy, report, X_test, y_test, cm


def random(X_train, y_train, X_test):
    from sklearn.ensemble import RandomForestClassifier
    classifier = RandomForestClassifier(n_estimators=10, criterion='entropy', random_state=0)
    classifier.fit(X_train, y_train)
    classifier, Accuracy, report, X_test, y_test, cm = cm_prediction(classifier, X_test)
    return classifier, Accuracy, report, X_test, y_test, cm


def forward_Classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf):
    dataframe = pd.DataFrame(
        index=['ForwardSelection'],
        columns=['Logistic', 'SVMl', 'SVMnl', 'KNN', 'Navie', 'Decision', 'Random']
    )
    for number, idex in enumerate(dataframe.index):
        dataframe.loc[idex, 'Logistic'] = acclog[number]
        dataframe.loc[idex, 'SVMl']     = accsvml[number]
        dataframe.loc[idex, 'SVMnl']    = accsvmnl[number]
        dataframe.loc[idex, 'KNN']      = accknn[number]
        dataframe.loc[idex, 'Navie']    = accnav[number]
        dataframe.loc[idex, 'Decision'] = accdes[number]
        dataframe.loc[idex, 'Random']   = accrf[number]
    return dataframe


In [2]:
dataset1 = pd.read_csv("prep.csv", index_col=None)
df2 = dataset1.copy()

df2 = pd.get_dummies(df2, drop_first=True)

indep_X = df2.drop('classification_yes', axis=1)
dep_Y = df2['classification_yes']

df2


,age,bp,al,su,bgr,bu,sc,sod,pot,hrmo,...,pc_normal,pcc_present,ba_present,htn_yes,dm_yes,cad_yes,appet_yes,pe_yes,ane_yes,classification_yes
0,2.000000,76.459948,3.0,0.0,148.112676,57.482105,3.077356,137.528754,4.627244,12.518156,...,False,False,False,False,False,False,True,True,False,True
1,3.000000,76.459948,2.0,0.0,148.112676,22.000000,0.700000,137.528754,4.627244,10.700000,...,True,False,False,False,False,False,True,False,False,True
2,4.000000,76.459948,1.0,0.0,99.000000,23.000000,0.600000,138.000000,4.400000,12.000000,...,True,False,False,False,False,False,True,False,False,True
3,5.000000,76.459948,1.0,0.0,148.112676,16.000000,0.700000,138.000000,3.200000,8.100000,...,True,False,False,False,False,False,True,False,True,True
4,5.000000,50.000000,0.0,0.0,148.112676,25.000000,0.600000,137.528754,4.627244,11.800000,...,True,False,False,False,False,False,True,False,False,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
394,51.492308,70.000000,0.0,0.0,219.000000,36.000000,1.300000,139.000000,3.700000,12.500000,...,True,False,False,False,False,False,True,False,False,True
395,51.492308,70.000000,0.0,2.0,220.000000,68.000000,2.800000,137.528754,4.627244,8.700000,...,True,False,False,True,True,False,True,False,True,True
396,51.492308,70.000000,3.0,0.0,110.000000,115.000000,6.000000,134.000000,2.700000,9.100000,...,True,False,False,True,True,False,False,False,False,True
397,51.492308,90.000000,0.0,0.0,207.000000,80.000000,6.800000,142.000000,5.500000,8.500000,...,True,False,False,True,True,False,True,False,True,True


In [7]:
# Apply Forward Selection with n=9 features
fs_feat, selected_indices = forward_selection(indep_X, dep_Y, 8)
print("Selected feature indices:", selected_indices)
print("Selected feature names:", list(indep_X.columns[selected_indices]))

acclog   = []
accsvml  = []
accsvmnl = []
accknn   = []
accnav   = []
accdes   = []
accrf    = []


C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Selected feature indices: [9, 6, 3, 2, 15, 0, 4, 5]
Selected feature names: ['hrmo', 'sc', 'su', 'al', 'sg_d', 'age', 'bgr', 'bu']


In [8]:
X_train, X_test, y_train, y_test = split_scalar(fs_feat, dep_Y)

classifier, Accuracy, report, X_test, y_test, cm = logistic(X_train, y_train, X_test)
acclog.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = svm_linear(X_train, y_train, X_test)
accsvml.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = svm_NL(X_train, y_train, X_test)
accsvmnl.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = knn(X_train, y_train, X_test)
accknn.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = Navie(X_train, y_train, X_test)
accnav.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = Decision(X_train, y_train, X_test)
accdes.append(Accuracy)

classifier, Accuracy, report, X_test, y_test, cm = random(X_train, y_train, X_test)
accrf.append(Accuracy)

result = forward_Classification(acclog, accsvml, accsvmnl, accknn, accnav, accdes, accrf)


In [6]:
result
#9

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ForwardSelection,0.98,1.0,0.99,0.98,0.92,0.96,1.0


In [9]:
result
#8

,Logistic,SVMl,SVMnl,KNN,Navie,Decision,Random
ForwardSelection,0.98,0.98,0.98,0.98,0.92,0.95,0.99


When the numbers of the feature is 9 , we got the best performence

In [12]:
X_arr = indep_X.values
X_train, X_test, y_train, y_test = train_test_split(
    X_arr, dep_Y, test_size=0.25, random_state=0
)

feature_names = indep_X.columns

selected_fwd = []
remaining_fwd = list(range(X_arr.shape[1]))
fwd_log = []

for step in range(9):
    best_acc = -np.inf
    best_col = None
    for col in remaining_fwd:
        candidate = selected_fwd + [col]
        clf = LogisticRegression(max_iter=1000, random_state=0)
        clf.fit(X_train[:, candidate], y_train)
        acc = accuracy_score(y_test, clf.predict(X_test[:, candidate]))
        if acc > best_acc:
            best_acc = acc
            best_col = col
    selected_fwd.append(best_col)
    remaining_fwd.remove(best_col)
    fwd_log.append({
        'Step'        : step + 1,
        'Feature Added': feature_names[best_col],
        'Accuracy'    : round(best_acc, 4)
    })

fwd_df = pd.DataFrame(fwd_log).set_index('Step')
print("Forward Selection — Step-by-step feature addition")
fwd_df

C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Anaconda3\envs\aiml\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = 

Forward Selection — Step-by-step feature addition


,Feature Added,Accuracy
Step,,
1,hrmo,0.91
2,sc,0.95
3,su,0.98
4,al,0.98
5,sg_d,1.00
6,age,1.00
7,bgr,1.00
8,bu,1.00
9,sod,1.00
